# 00 - Setup and data check
Checks how much Sentinel-1 and Sentinel-2 data exists over the Hyderabad AOI for 2019 and 2025.
Use the result to choose **one Sentinel-1 orbit direction** and confirm the years in `config.yaml`.

In [ ]:
# --- Setup (run once per Colab session) ---
!git clone -q https://github.com/veeresh0804/PrithviBodh.git 2>/dev/null || (cd PrithviBodh && git pull -q)
%cd /content/PrithviBodh
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/PrithviBodh'      # Earth Engine exports land here
import os; os.makedirs(DRIVE, exist_ok=True)

import sys; sys.path.insert(0, '.')
from prithvibodh.config import load_config
cfg = load_config()
cfg['gee_project'] = 'YOUR_EE_PROJECT_ID'         # <-- your free noncommercial project
from prithvibodh import gee_data as G
G.init(cfg)

In [ ]:
rep = G.availability_report(cfg)
rep.to_csv(f'{DRIVE}/data_availability.csv', index=False)
rep

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for y in cfg['years']:
    d = rep[rep.year == y]
    ax[0].plot(d.month, d.s1_ascending, 'o--', label=f'{y} asc')
    ax[0].plot(d.month, d.s1_descending, 'o-', label=f'{y} desc')
    ax[1].plot(d.month, d.s2_under_20pct_cloud, 'o-', label=str(y))
ax[0].set_title('Sentinel-1 scenes / month'); ax[1].set_title('Sentinel-2 scenes <20% cloud / month')
for a in ax: a.set_xticks(range(1, 13)); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{DRIVE}/data_availability.png', dpi=150)

In [ ]:
# Cloud problem in numbers: mean clear Sentinel-2 observations per pixel, per season
import ee
region = G.aoi(cfg)
for season, months in cfg['seasons'].items():
    n = G.s2_clear_count(cfg, region, cfg['train_year'], months).reduceRegion(
        ee.Reducer.mean(), region, scale=100, maxPixels=1e9).getInfo()
    print(f"{season:13s} mean clear observations: {n['clear_obs']:.1f}")

**Next:** pick the orbit (ASCENDING or DESCENDING) with steady scenes every month in both years,
write it into `config.yaml` -> `sentinel1.orbit`, and commit.